In [10]:
import threading
import time
import random
import tkinter as tk

brightness = 0.0
light_on = False
threshold = 37.0
running = True

lock = threading.Lock()


def sensor_thread():
    global brightness, running
    while running:
        new_value = random.uniform(0, 100)
        with lock:
            brightness = new_value
        time.sleep(0.35)


def controller_thread():
    global light_on, running
    while running:
        with lock:
            current_brightness = brightness
            light_on = current_brightness < threshold
        time.sleep(0.2)


def launch_dashboard():
    root = tk.Tk()
    root.title("Light Controller Dashboard")
    root.geometry("420x320")
    root.configure(bg="#f5f5f5")

    title = tk.Label(
        root,
        bg="#f5f5f5"
    )
    title.pack

    value_label = tk.Label(
        root,
        text="",
        bg="#f5f5f5",
    )
    value_label.pack(pady=6)

    canvas = tk.Canvas(root, width=320, height=180, bg="white", highlightthickness=1, highlightbackground="#dddddd")
    canvas.pack(pady=8)

    square = canvas.create_rectangle(40, 40, 120, 120, fill="brown", outline="#000000", width=2)
    square_text = canvas.create_text(80, 132, text="Brightness", font=("Arial", 10))

    circle = canvas.create_oval(200, 40, 280, 120, fill="grey", outline="#000000", width=2)
    circle_text = canvas.create_text(240, 132, text="Light", font=("Arial", 10))

    def refresh_ui():
        with lock:
            b = brightness
            l = light_on

        is_bright = b >= threshold

        square_color = "red" if is_bright else "brown"
        circle_color = "yellow" if l else "grey"

        canvas.itemconfig(square, fill=square_color)
        canvas.itemconfig(circle, fill=circle_color)

        value_label.config(
            text=(
                f"brightness : {b:6.2f}\n"
                f"threshold  : {threshold:6.2f}\n"
                f"light_on   : {l}"
            )
        )

        root.after(150, refresh_ui)

    def on_close():
        global running
        running = False
        root.destroy()

    root.protocol("WM_DELETE_WINDOW", on_close)
    refresh_ui()
    root.mainloop()


sensor = threading.Thread(target=sensor_thread, daemon=True)
controller = threading.Thread(target=controller_thread, daemon=True)

sensor.start()
controller.start()

launch_dashboard()

running = False
sensor.join()
controller.join()